# Islamic Mute: train the voice model (keep speech, recitation and singing; remove instruments)

Fine-tunes DeepFilterNet3 (MIT/Apache) so that **all voice is kept** and only musical instruments are removed,
then exports the 30 ms streaming ONNX model used by the Linux/Android engine and scores it against the
baseline on our evaluation set.

| Data | Licence |
|---|---|
| LibriSpeech (attached) | CC BY 4.0 |
| Arabic Speech Corpus (Hugging Face) | CC BY 4.0 |
| VocalSet singing (Hugging Face) | CC BY 4.0 |
| Diverse Quran Reciters (attached) | rights belong to the reciters: get permission before releasing weights, or set `USE_QURAN = False` |
| Slakh2100 instrument stems and mixes (Hugging Face subset) | CC BY 4.0 |

Outputs (in `/kaggle/working/out`): `voice_v1.onnx` + `voice_v1.state_init.bin` (drop-in model), `last.pt`, `eval_summary.md`.

In [ ]:
# ---- settings -------------------------------------------------------------
TRAIN_HOURS = 5.0            # wall-clock training budget (Kaggle GPU sessions stop at 12 h)
DATA_HOURS = "speech_en=3,speech_ar=3,singing=4,recitation=4,slakh=10"
USE_QURAN = True             # see the licence note above
BATCH = 16
MAX_STEPS = 60000            # upper bound; TRAIN_HOURS normally stops first
VOICE_WEIGHTS = {"speech_en": 0.7, "speech_ar": 1.0, "singing": 1.5, "recitation": 1.5}

import os, subprocess, sys, time
CODE, TMP, OUT = "/kaggle/working/code", "/kaggle/tmp", "/kaggle/working/out"
for d in (CODE, TMP, OUT):
    os.makedirs(d, exist_ok=True)

def run(cmd, cwd=CODE, env=None):
    """Run a shell command, stream its output, fail loudly."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

In [ ]:
import os
for d in ['/kaggle/working/code/eval', '/kaggle/working/code/kaggle', '/kaggle/working/code/models/dfn3', '/kaggle/working/code/models/voice', '/kaggle/working/code/testset']:
    os.makedirs(d, exist_ok=True)

## Source files (generated from the repository)

In [ ]:
%%writefile /kaggle/working/code/models/dfn3/df_compat.py
"""Import shims so the official `deepfilternet` 0.5.6 package works on newer torch/torchaudio.

torchaudio >= 2.5 removed `torchaudio.backend.common`, which `df.io` imports at module load.
Import this module before `df`.
"""
import sys
import types

try:
    import torchaudio.backend.common  # noqa: F401
except Exception:  # pragma: no cover - depends on the installed torchaudio
    backend = sys.modules.get("torchaudio.backend") or types.ModuleType("torchaudio.backend")
    common = types.ModuleType("torchaudio.backend.common")

    class AudioMetaData:  # only used for type hints in df.io
        pass

    common.AudioMetaData = AudioMetaData
    backend.common = common
    sys.modules["torchaudio.backend"] = backend
    sys.modules["torchaudio.backend.common"] = common

In [ ]:
%%writefile /kaggle/working/code/models/dfn3/export_stream.py
"""Export DeepFilterNet3 as a streaming, stateful ONNX model for the engine.

Engine model contract (every model the engine loads follows it):
  inputs : audio_in  float32 [hop]         one hop of mono audio at the model rate
           state_in  float32 [state_size]  opaque state, zeros at start
  outputs: audio_out float32 [hop]
           state_out float32 [state_size]
  metadata_props: im.contract=1, im.sample_rate, im.hop, im.state_size,
                  im.latency_samples, im.model_name, im.licence

All DSP (STFT, features, ISTFT) lives inside the graph, so the engine only
loops hops and threads state. This file uses the official `deepfilternet`
package (PyPI) for weights and layers and adds per-frame caches for the
causal convolutions, the GRU hidden states and the 2-frame look-ahead.
"""
import argparse
import inspect
import math
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import numpy as np
import torch
import torch.nn as nn

import df_compat  # noqa: F401  (newer torchaudio)
from df.enhance import init_df
from df.deepfilternet3 import ModelParams

PAD_TYPES = (nn.ConstantPad2d,)


def run_nopad(seq: nn.Sequential, x: torch.Tensor) -> torch.Tensor:
    """Run a Conv*NormAct Sequential without its causal time padding.

    The caller supplies the past frames explicitly instead of zero padding."""
    for layer in seq:
        if isinstance(layer, PAD_TYPES):
            continue
        x = layer(x)
    return x


def norm_alpha(sr: int, hop: int, tau: float) -> float:
    """Same rounding as libDF calc_norm_alpha."""
    alpha = math.exp(-(hop / sr) / tau)
    a, precision = 1.0, 3
    while a >= 1.0:
        a = round(alpha * 10**precision) / 10**precision
        precision += 1
    return a


class StreamDFN3(nn.Module):
    def __init__(self, model: nn.Module, df_state):
        super().__init__()
        p = ModelParams()
        self.m = model.eval()
        self.sr = df_state.sr()
        self.hop = df_state.hop_size()
        self.n_fft = df_state.fft_size()
        self.nf = self.n_fft // 2 + 1
        self.nb_erb = p.nb_erb
        self.nb_df = p.nb_df
        self.order = p.df_order
        assert p.conv_lookahead == p.df_lookahead == 2, "export assumes 2-frame look-ahead"
        assert self.n_fft == 2 * self.hop
        self.la = p.df_lookahead
        self.alpha = norm_alpha(self.sr, self.hop, p.norm_tau)
        ch = p.conv_ch
        kt = p.df_pathway_kernel_size_t
        self.kt = kt
        self.ch = ch

        widths = df_state.erb_widths()
        fb = torch.zeros(self.nf, self.nb_erb)
        start = 0
        for i, w in enumerate(widths.tolist()):
            fb[start:start + w, i] = 1.0 / w  # mean power per band (libDF compute_band_corr)
            start += w
        self.register_buffer("erb_mean_fb", fb)

        n = torch.arange(self.n_fft, dtype=torch.float64)
        win = torch.sin(0.5 * math.pi * torch.sin(math.pi * (n + 0.5) / self.n_fft) ** 2)
        self.register_buffer("win", win.float())
        wnorm = 1.0 / (self.n_fft ** 2 / (2 * self.hop))
        k = torch.arange(self.nf, dtype=torch.float64)
        ang = 2 * math.pi * torch.outer(n, k) / self.n_fft  # [N, F]
        self.register_buffer("fwd_re", (torch.cos(ang) * wnorm).float())
        self.register_buffer("fwd_im", (-torch.sin(ang) * wnorm).float())
        ck = torch.full((self.nf,), 2.0, dtype=torch.float64)
        ck[0] = 1.0
        ck[-1] = 1.0
        self.register_buffer("inv_re", (torch.cos(ang).T * ck[:, None]).float())  # [F, N]
        self.register_buffer("inv_im", (-torch.sin(ang).T * ck[:, None]).float())

        erb_init = torch.linspace(-60.0, -90.0, self.nb_erb)
        unit_init = torch.linspace(0.001, 0.0001, self.nb_df)
        gru_h = 256
        n_enc = self.m.enc.emb_gru.gru.num_layers
        n_erb = self.m.erb_dec.emb_gru.gru.num_layers
        n_df = self.m.df_dec.df_gru.gru.num_layers
        assert self.m.enc.emb_gru.gru.hidden_size == gru_h

        # name -> shape; order defines the flat state layout
        self.layout = [
            ("ana_mem", (self.n_fft - self.hop,)),
            ("syn_mem", (self.n_fft - self.hop,)),
            ("erb_norm", (self.nb_erb,)),
            ("unit_norm", (self.nb_df,)),
            ("erb_hist", (2, self.nb_erb)),          # conv_kernel_inp time 3 -> 2 past frames
            ("specf_hist", (2, 2, self.nb_df)),      # [T, re/im, F]
            ("c0_hist", (kt - 1, ch, self.nb_df)),   # df_convp time kernel kt
            ("spec_hist", (2 * self.la, 2, self.nf)),  # spec frames n-4..n-1, [T, re/im, F]
            ("h_enc", (n_enc, 1, gru_h)),
            ("h_erb", (n_erb, 1, gru_h)),
            ("h_df", (n_df, 1, gru_h)),
        ]
        self.offsets = {}
        off = 0
        for name, shape in self.layout:
            size = int(np.prod(shape))
            self.offsets[name] = (off, size, shape)
            off += size
        self.state_size = off
        init = torch.zeros(self.state_size)
        o, s, _ = self.offsets["erb_norm"]
        init[o:o + s] = erb_init
        o, s, _ = self.offsets["unit_norm"]
        init[o:o + s] = unit_init
        self.register_buffer("state_init", init)
        # algorithmic delay: STFT overlap (n_fft - hop) + look-ahead frames
        self.latency_samples = (self.n_fft - self.hop) + self.la * self.hop

    def get(self, state, name):
        o, s, shape = self.offsets[name]
        return state[o:o + s].reshape(shape)

    def forward(self, audio_in: torch.Tensor, state_in: torch.Tensor):
        m = self.m
        a = self.alpha
        ana_mem = self.get(state_in, "ana_mem")
        syn_mem = self.get(state_in, "syn_mem")
        erb_norm = self.get(state_in, "erb_norm")
        unit_norm = self.get(state_in, "unit_norm")
        erb_hist = self.get(state_in, "erb_hist")
        specf_hist = self.get(state_in, "specf_hist")
        c0_hist = self.get(state_in, "c0_hist")
        spec_hist = self.get(state_in, "spec_hist")
        h_enc = self.get(state_in, "h_enc")
        h_erb = self.get(state_in, "h_erb")
        h_df = self.get(state_in, "h_df")

        # analysis
        frame = torch.cat([ana_mem, audio_in]) * self.win
        re = frame @ self.fwd_re
        im = frame @ self.fwd_im

        # features (libDF erb + band_mean_norm_erb, band_unit_norm)
        erb = (re * re + im * im) @ self.erb_mean_fb
        erb_db = 10.0 * torch.log10(erb + 1e-10)
        erb_norm_new = erb_db * (1 - a) + erb_norm * a
        feat_erb = (erb_db - erb_norm_new) / 40.0
        re_l = re[: self.nb_df]
        im_l = im[: self.nb_df]
        mag = torch.sqrt(re_l * re_l + im_l * im_l)
        unit_new = mag * (1 - a) + unit_norm * a
        sq = torch.sqrt(unit_new)
        feat_spec = torch.stack([re_l / sq, im_l / sq])  # [2, F']

        # encoder (convs see frames n-2..n, i.e. the 2-frame look-ahead)
        fe = torch.cat([erb_hist, feat_erb[None]]).reshape(1, 1, 3, self.nb_erb)
        e0 = run_nopad(m.enc.erb_conv0, fe)
        e1 = m.enc.erb_conv1(e0)
        e2 = m.enc.erb_conv2(e1)
        e3 = m.enc.erb_conv3(e2)
        fs = torch.cat([specf_hist, feat_spec[None]]).permute(1, 0, 2).reshape(1, 2, 3, self.nb_df)
        c0 = run_nopad(m.enc.df_conv0, fs)  # [1, ch, 1, F']
        c1 = m.enc.df_conv1(c0)
        cemb = m.enc.df_fc_emb(c1.permute(0, 2, 3, 1).flatten(2))
        emb = m.enc.combine(e3.permute(0, 2, 3, 1).flatten(2), cemb)
        emb, h_enc_new = m.enc.emb_gru(emb, h_enc)

        # ERB decoder
        emb_d, h_erb_new = m.erb_dec.emb_gru(emb, h_erb)
        f8 = e3.shape[3]
        emb_d = emb_d.view(1, 1, f8, -1).permute(0, 3, 1, 2)
        d3 = m.erb_dec.convt3(m.erb_dec.conv3p(e3) + emb_d)
        d2 = m.erb_dec.convt2(m.erb_dec.conv2p(e2) + d3)
        d1 = m.erb_dec.convt1(m.erb_dec.conv1p(e1) + d2)
        mask = m.erb_dec.conv0_out(m.erb_dec.conv0p(e0) + d1)  # [1,1,1,E]

        # DF decoder
        c, h_df_new = m.df_dec.df_gru(emb, h_df)
        if m.df_dec.df_skip is not None:
            c = c + m.df_dec.df_skip(emb)
        c0_seq = torch.cat([c0_hist, c0[0, :, 0][None]]).permute(1, 0, 2).reshape(1, self.ch, self.kt, self.nb_df)
        c0p = run_nopad(m.df_dec.df_convp, c0_seq).permute(0, 2, 3, 1)  # [1,1,F',O*2]
        c = m.df_dec.df_out(c).view(1, 1, self.nb_df, self.order * 2) + c0p
        coefs = c.view(self.nb_df, self.order, 2)  # [F', O, re/im]

        # apply to spectrum frame n-2
        spec_cur = torch.stack([re, im])[None]  # [1, 2, F]
        spec_win = torch.cat([spec_hist, spec_cur])  # [5, 2, F] frames n-4..n
        target = spec_win[self.la]  # [2, F]
        gain = mask.reshape(1, self.nb_erb) @ m.mask.erb_inv_fb  # [1, F]
        out_re = target[0] * gain[0]
        out_im = target[1] * gain[0]
        sw = spec_win[:, :, : self.nb_df]  # [O, 2, F']
        cr = coefs[:, :, 0].T  # [O, F']
        ci = coefs[:, :, 1].T
        df_re = (sw[:, 0] * cr - sw[:, 1] * ci).sum(0)
        df_im = (sw[:, 0] * ci + sw[:, 1] * cr).sum(0)
        out_re = torch.cat([df_re, out_re[self.nb_df:]])
        out_im = torch.cat([df_im, out_im[self.nb_df:]])

        # synthesis
        y = (out_re @ self.inv_re + out_im @ self.inv_im) * self.win
        hop = self.hop
        audio_out = y[:hop] + syn_mem
        new_state = torch.cat([
            audio_in,
            y[hop:],
            erb_norm_new,
            unit_new,
            torch.cat([erb_hist[1:], feat_erb[None]]).flatten(),
            torch.cat([specf_hist[1:], feat_spec[None]]).flatten(),
            torch.cat([c0_hist[1:], c0[0, :, 0][None]]).flatten(),
            spec_win[1:].flatten(),
            h_enc_new.flatten(),
            h_erb_new.flatten(),
            h_df_new.flatten(),
        ])
        return audio_out, new_state


def build(checkpoint: str = ""):
    model, df_state, _ = init_df(log_level="ERROR")
    if checkpoint:  # fine-tuned weights from models/voice/train.py
        model.load_state_dict(torch.load(checkpoint, map_location="cpu")["model"])
    model = model.cpu().eval()
    return StreamDFN3(model, df_state), model, df_state


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="dfn3_stream.onnx")
    ap.add_argument("--checkpoint", default="", help="fine-tuned weights (models/voice/train.py)")
    ap.add_argument("--name", default="DeepFilterNet3 (speech mode)")
    ap.add_argument("--keeps", default="speech")
    args = ap.parse_args()
    s, _, _ = build(args.checkpoint)
    x = torch.zeros(s.hop)
    st = s.state_init.clone()
    with torch.no_grad():
        s(x, st)
        torch.onnx.export(
            s, (x, st), args.out, input_names=["audio_in", "state_in"],
            output_names=["audio_out", "state_out"], opset_version=17, do_constant_folding=True,
            **({"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}),
        )
    import onnx
    mp = onnx.load(args.out)
    meta = {
        "im.contract": "1", "im.sample_rate": str(s.sr), "im.hop": str(s.hop),
        "im.state_size": str(s.state_size), "im.latency_samples": str(s.latency_samples),
        "im.model_name": args.name, "im.licence": "MIT OR Apache-2.0",
        "im.keeps": args.keeps,
    }
    for k, v in meta.items():
        e = mp.metadata_props.add()
        e.key, e.value = k, v
    onnx.save(mp, args.out)
    s.state_init.numpy().astype("<f4").tofile(args.out.replace(".onnx", ".state_init.bin"))
    print(f"exported {args.out}: state_size={s.state_size} latency={s.latency_samples} samples")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/models/voice/train.py
"""Fine-tune DeepFilterNet3 so that ALL voice is kept (speech, Quran recitation, singing/nasheed)
and only musical instruments are removed.

Data: folders of audio files (wav/flac/mp3, any rate; resampled to 48 kHz on load).
  --voice DIR[:weight]   clean voice: speech, recitation, singing (target = this signal)
  --music DIR[:weight]   instrumental music / instrument stems (removed)
  --synth-music P        share of music examples generated on the fly by testset/build_testset.synth_music
Mixtures are made on the fly: random crop, random gain, music-to-voice ratio in [mvr_min, mvr_max] dB,
plus voice-only (keep everything) and music-only (output silence) examples.

Runs on GPU (Kaggle/Colab T4 is enough) or CPU for smoke tests:
  python models/voice/train.py --voice data/voice --music data/music --steps 20000 --out runs/v1
Export the result with:  python models/dfn3/export_stream.py --checkpoint runs/v1/last.pt --out voice_v1.onnx
"""
import argparse, glob, math, os, random, subprocess, sys, time

import numpy as np
import soundfile as sf
import torch
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(os.path.dirname(HERE))
sys.path.insert(0, os.path.join(ROOT, "models", "dfn3"))
sys.path.insert(0, os.path.join(ROOT, "testset"))
import df_compat  # noqa: E402,F401  (newer torchaudio)
from export_stream import norm_alpha  # noqa: E402

SR = 48000
AUDIO_EXT = (".wav", ".flac", ".mp3", ".ogg", ".m4a", ".opus")


# ---------------------------------------------------------------- data
def list_audio(spec):
    path, _, w = spec.partition(":")
    files = [f for f in glob.glob(os.path.join(path, "**", "*"), recursive=True) if f.lower().endswith(AUDIO_EXT)]
    return files, float(w or 1.0)


def load_48k_mono(path):
    if path.lower().endswith((".wav", ".flac", ".ogg")):
        x, sr = sf.read(path, dtype="float32", always_2d=True)
        x = x.mean(1)
        if sr != SR:
            x = torch.from_numpy(x)
            import torchaudio
            x = torchaudio.functional.resample(x, sr, SR).numpy()
        return x
    out = subprocess.run(["ffmpeg", "-v", "error", "-i", path, "-ac", "1", "-ar", str(SR), "-f", "f32le", "-"],
                         check=True, capture_output=True).stdout
    return np.frombuffer(out, np.float32).copy()


class Pool:
    """Weighted file pool. 48 kHz wav/flac files are read partially (fast); other files are
    decoded fully and cached (prepare data with kaggle/prepare_data.py to avoid that)."""

    def __init__(self, specs, cache=64):
        self.groups = [list_audio(s) for s in specs]
        self.groups = [(f, w) for f, w in self.groups if f]
        self.cache, self.max = {}, cache
        self.info = {}

    def __bool__(self):
        return bool(self.groups)

    def sample(self, n, rng):
        files, _ = rng.choices(self.groups, weights=[w for _, w in self.groups])[0]
        f = rng.choice(files)
        if f.lower().endswith((".wav", ".flac")):
            if f not in self.info:
                i = sf.info(f)
                self.info[f] = (i.frames, i.samplerate)
            frames, sr = self.info[f]
            if sr == SR and frames >= n:
                s = rng.randint(0, frames - n)
                x, _ = sf.read(f, start=s, frames=n, dtype="float32", always_2d=True)
                return x.mean(1)
        if f not in self.cache:
            if len(self.cache) >= self.max:
                self.cache.pop(next(iter(self.cache)))
            self.cache[f] = load_48k_mono(f)
        x = self.cache[f]
        if len(x) < n:
            x = np.tile(x, int(math.ceil(n / max(len(x), 1))) + 1)
        s = rng.randint(0, len(x) - n)
        return x[s:s + n]


def rms(x):
    return float(np.sqrt(np.mean(x ** 2) + 1e-10))


def make_batch(voice, music, args, rng):
    n = int(args.seconds * SR)
    xs, ys = [], []
    for _ in range(args.batch):
        r = rng.random()
        v = voice.sample(n, rng) if r > args.p_music_only else np.zeros(n, np.float32)
        if r > 1 - args.p_voice_only:
            m = np.zeros(n, np.float32)
        else:
            if music and rng.random() > args.synth_music:
                m = music.sample(n, rng)
            else:
                from build_testset import synth_music
                m = synth_music(args.seconds, rng.randint(0, 10 ** 9), lead=rng.random() < 0.6).mean(1)[:n]
            if v.any():
                m = m / rms(m) * rms(v) * 10 ** (rng.uniform(args.mvr_min, args.mvr_max) / 20)
        mix = v + m
        g = 10 ** (rng.uniform(-30, -6) / 20) / (np.max(np.abs(mix)) + 1e-6)  # random level, never clipping
        xs.append(mix * g)
        ys.append(v * g)
    return torch.from_numpy(np.stack(xs)).float(), torch.from_numpy(np.stack(ys)).float()


class Batches(torch.utils.data.IterableDataset):
    """Endless stream of (mixture, voice) batches, built in DataLoader worker processes."""

    def __init__(self, args):
        self.args = args

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        wid = info.id if info else 0
        rng = random.Random(self.args.seed * 1000 + wid + int(time.time()))
        voice, music = Pool(self.args.voice), Pool(self.args.music)
        while True:
            yield make_batch(voice, music, self.args, rng)


# ---------------------------------------------------------------- DFN front end (batched, matches libDF / the streaming export)
class FrontEnd(torch.nn.Module):
    def __init__(self, df_state, nb_df, device):
        super().__init__()
        self.hop, self.n_fft = df_state.hop_size(), df_state.fft_size()
        self.nb_df = nb_df
        self.alpha = norm_alpha(SR, self.hop, 1.0)
        n = torch.arange(self.n_fft, dtype=torch.float64)
        self.win = torch.sin(0.5 * math.pi * torch.sin(math.pi * (n + 0.5) / self.n_fft) ** 2).float().to(device)
        self.wnorm = 1.0 / (self.n_fft ** 2 / (2 * self.hop))
        widths = df_state.erb_widths()
        nf = self.n_fft // 2 + 1
        fb = torch.zeros(nf, len(widths))
        s = 0
        for i, w in enumerate(widths.tolist()):
            fb[s:s + w, i] = 1.0 / w
            s += w
        self.erb_fb = fb.to(device)
        self.erb_init = torch.linspace(-60.0, -90.0, len(widths), device=device)
        self.unit_init = torch.linspace(0.001, 0.0001, nb_df, device=device)

    def stft(self, x):  # x [B, T] -> complex [B, frames, F]; frame k = samples [(k-1)hop, (k+1)hop)
        x = F.pad(x, (self.hop, 0))
        fr = x.unfold(-1, self.n_fft, self.hop) * self.win
        return torch.fft.rfft(fr, dim=-1) * self.wnorm

    def istft(self, spec, length):
        fr = torch.fft.irfft(spec, n=self.n_fft, dim=-1) * self.n_fft * self.win  # unnormalised inverse like libDF
        b, t, _ = fr.shape
        out = F.fold(fr.transpose(1, 2), output_size=(1, (t + 1) * self.hop), kernel_size=(1, self.n_fft),
                     stride=(1, self.hop)).reshape(b, -1)
        return out[:, self.hop:self.hop + length]

    def features(self, spec):
        a = self.alpha
        erb = 10 * torch.log10((spec.abs() ** 2) @ self.erb_fb + 1e-10)  # [B,T,E]
        mag = spec[..., : self.nb_df].abs()
        st_e = self.erb_init.expand(spec.shape[0], -1)
        st_u = self.unit_init.expand(spec.shape[0], -1)
        fe, fu = [], []
        for t in range(spec.shape[1]):
            st_e = erb[:, t] * (1 - a) + st_e * a
            st_u = mag[:, t] * (1 - a) + st_u * a
            fe.append((erb[:, t] - st_e) / 40.0)
            fu.append(spec[:, t, : self.nb_df] / torch.sqrt(st_u))
        feat_erb = torch.stack(fe, 1).unsqueeze(1)  # [B,1,T,E]
        feat_spec = torch.view_as_real(torch.stack(fu, 1)).unsqueeze(1)  # [B,1,T,F',2]
        return feat_erb, feat_spec


# ---------------------------------------------------------------- losses (DeepFilterNet-style)
def spec_loss(est, ref, gamma=0.3):
    em, rm = est.abs().clamp_min(1e-12), ref.abs().clamp_min(1e-12)
    l_mag = F.mse_loss(em ** gamma, rm ** gamma)
    ec = em ** gamma * est / em
    rc = rm ** gamma * ref / rm
    l_cpx = F.mse_loss(torch.view_as_real(ec), torch.view_as_real(rc))
    return l_mag + l_cpx


def multires_loss(est_wav, ref_wav, ffts=(256, 512, 1024, 2048)):
    loss = 0.0
    for n in ffts:
        w = torch.hann_window(n, device=est_wav.device)
        e = torch.stft(est_wav, n, n // 4, window=w, return_complex=True)
        r = torch.stft(ref_wav, n, n // 4, window=w, return_complex=True)
        loss = loss + spec_loss(e, r)
    return loss / len(ffts)


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--voice", action="append", default=[])
    ap.add_argument("--music", action="append", default=[])
    ap.add_argument("--synth-music", type=float, default=0.3)
    ap.add_argument("--steps", type=int, default=20000)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--seconds", type=float, default=3.0)
    ap.add_argument("--lr", type=float, default=5e-5)
    ap.add_argument("--mvr-min", type=float, default=-15.0)
    ap.add_argument("--mvr-max", type=float, default=10.0)
    ap.add_argument("--p-voice-only", type=float, default=0.1)
    ap.add_argument("--p-music-only", type=float, default=0.1)
    ap.add_argument("--out", default="runs/voice")
    ap.add_argument("--resume", default="")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--log-every", type=int, default=50)
    ap.add_argument("--save-every", type=int, default=1000)
    ap.add_argument("--hours", type=float, default=0, help="stop and save after this much wall time (0 = no limit)")
    ap.add_argument("--workers", type=int, default=4)
    args = ap.parse_args()

    from df.enhance import init_df
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, df_state, _ = init_df(log_level="ERROR")  # pretrained DeepFilterNet3 = starting point
    model = model.to(device).train()

    def freeze_bn():  # keep pretrained BatchNorm statistics; small fine-tuning batches would corrupt them
        for m in model.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()

    freeze_bn()
    if args.resume:
        model.load_state_dict(torch.load(args.resume, map_location=device)["model"])
    fe = FrontEnd(df_state, model.nb_df, device)
    if not Pool(args.voice):
        sys.exit("no voice files found (--voice DIR)")
    loader = iter(torch.utils.data.DataLoader(Batches(args), batch_size=None, num_workers=args.workers,
                                              prefetch_factor=4 if args.workers else None,
                                              persistent_workers=bool(args.workers)))
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)

    def lr_at(step, elapsed):
        """Warmup, then cosine decay over whichever ends first: the step budget or the time budget."""
        frac = step / args.steps
        if args.hours:
            frac = max(frac, elapsed / (args.hours * 3600))
        frac = min(frac, 1.0)
        warm = min(1.0, step / max(1, min(500, args.steps // 20)))
        return args.lr * warm * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * frac)))
    os.makedirs(args.out, exist_ok=True)
    t0, run = time.time(), 0.0
    for step in range(1, args.steps + 1):
        x, y = next(loader)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        X, Y = fe.stft(x), fe.stft(y)
        feat_erb, feat_spec = fe.features(X)
        out = model(torch.view_as_real(X).unsqueeze(1), feat_erb, feat_spec)[0]
        E = torch.view_as_complex(out.squeeze(1).contiguous())
        e_wav = fe.istft(E, y.shape[-1])
        loss = spec_loss(E, Y) * 1000 + multires_loss(e_wav, y) * 500
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        lr = lr_at(step, time.time() - t0)
        for g in opt.param_groups:
            g["lr"] = lr
        opt.step()
        run = 0.98 * run + 0.02 * loss.item() if step > 1 else loss.item()
        if step % args.log_every == 0 or step == 1:
            print(f"step {step}/{args.steps} loss {run:.4f} lr {lr:.2e} {time.time()-t0:.0f}s", flush=True)
        out_of_time = args.hours and time.time() - t0 > args.hours * 3600
        if step % args.save_every == 0 or step == args.steps or out_of_time:
            torch.save({"model": model.state_dict(), "step": step, "args": vars(args)}, os.path.join(args.out, "last.pt"))
        if out_of_time:
            print(f"time budget reached at step {step}", flush=True)
            break
    print("done:", os.path.join(args.out, "last.pt"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/models/voice/test_frontend.py
"""The batched training front end must match libDF (and therefore the streaming export)."""
import numpy as np, torch
from libdf import erb, erb_norm, unit_norm
from df.enhance import init_df
from train import FrontEnd

model, st, _ = init_df(log_level="ERROR")
fe = FrontEnd(st, model.nb_df, "cpu")
rng = np.random.default_rng(0)
x = (0.1 * rng.standard_normal(48000 * 2)).astype(np.float32)
ref = st.analysis(x[None])[0]
X = fe.stft(torch.from_numpy(x[None]))[0].numpy()
n = min(len(ref), len(X))
print("stft max err", np.abs(ref[:n] - X[:n]).max())
fe_erb, fe_spec = fe.features(torch.from_numpy(X[None]))
r_erb = erb_norm(erb(ref[None], st.erb_widths()), fe.alpha)[0]
r_un = unit_norm(ref[None, :, : model.nb_df], fe.alpha)[0]
print("erb feat max err", np.abs(fe_erb[0, 0].numpy()[:n] - r_erb[:n]).max())
print("unit feat max err", np.abs(torch.view_as_complex(fe_spec[0, 0]).numpy()[:n] - r_un[:n]).max())
y = fe.istft(torch.from_numpy(X[None]), len(x))[0].numpy()
err = np.abs(y[960:-960] - x[960:-960]).max()
print("istft(stft(x)) reconstruction max err", err)
assert err < 1e-4
print("FRONTEND OK")

In [ ]:
%%writefile /kaggle/working/code/testset/fetch_sources.py
"""Download a small, licensed set of voice clips for the evaluation test set.

Each source is a public Hugging Face dataset read through the datasets-server
rows API, so only the needed clips are downloaded. Licences are recorded in
data/sources/SOURCES.json. Clips are used for local evaluation only.
"""
import base64, io, json, os, subprocess, sys, tempfile, time, urllib.parse, urllib.request

import numpy as np
import soundfile as sf

OUT = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data", "sources")
API = "https://datasets-server.huggingface.co/rows"

SOURCES = [
    # (category, dataset, config, split, offsets, licence, note)
    ("speech_en", "sanchit-gandhi/librispeech_asr_dummy", "default", "test.clean", [0, 3, 7, 12], "CC-BY-4.0", "LibriSpeech"),
    ("speech_ar", "tunis-ai/arabic_speech_corpus", "default", "train", [0, 50, 100, 150], "CC-BY-4.0", "Arabic Speech Corpus (Halabi)"),
    ("recitation", "fawazahmed0/quran-audio", "default", "train", [0, 1, 4000, 9000, 30000, 60000], "CC0-1.0 (dataset tag; recordings belong to reciters)", "Quran audio"),
    ("singing", "Bill13579/vocalset-mirror", "default", "train", [0, 200, 900, 1500, 2300, 3100], "CC-BY-4.0", "VocalSet"),
]


def get_json(url, tries=6):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return json.loads(r.read().decode())
        except Exception as e:  # rate limits and cold caches are common
            print(f"  retry {i+1}: {e}", file=sys.stderr)
            time.sleep(3 * (i + 1))
    return None


def fetch(url):
    with urllib.request.urlopen(url, timeout=120) as r:
        return r.read()


def decode(blob):
    """Decode any audio container with ffmpeg (mp3/flac/wav/ogg) to mono float32."""
    with tempfile.TemporaryDirectory() as d:
        src, dst = os.path.join(d, "in.bin"), os.path.join(d, "out.wav")
        open(src, "wb").write(blob)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", src, "-ac", "1", "-c:a", "pcm_f32le", dst], check=True)
        data, sr = sf.read(dst, dtype="float32")
    return data, sr


def main():
    os.makedirs(OUT, exist_ok=True)
    meta_path = os.path.join(OUT, "SOURCES.json")
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
    for cat, ds, cfg, split, offsets, lic, note in SOURCES:
        for off in offsets:
            name = f"{cat}_{off}.wav"
            path = os.path.join(OUT, name)
            if os.path.exists(path):
                continue
            q = urllib.parse.urlencode({"dataset": ds, "config": cfg, "split": split, "offset": off, "length": 1})
            d = get_json(f"{API}?{q}")
            if not d or not d.get("rows"):
                print(f"skip {ds}@{off}: {str(d)[:120]}")
                continue
            row = d["rows"][0]["row"]
            audio = row.get("audio")
            if isinstance(audio, list):
                audio = audio[0]
            try:
                if isinstance(audio, dict) and audio.get("src"):
                    blob = fetch(audio["src"])
                elif isinstance(audio, dict) and audio.get("bytes"):
                    blob = base64.b64decode(audio["bytes"])
                else:
                    print(f"skip {ds}@{off}: no audio field")
                    continue
                mono, sr = decode(blob)
            except Exception as e:
                print(f"skip {ds}@{off}: {e}")
                continue
            sf.write(path, mono, sr, subtype="FLOAT")
            text = row.get("text") or row.get("transcription") or row.get("sentence") or ""
            meta[name] = {"category": cat, "dataset": ds, "offset": off, "licence": lic, "source": note,
                          "sample_rate": sr, "seconds": round(len(mono) / sr, 2), "text": text}
            print(f"got {name}: {len(mono)/sr:.1f}s @ {sr} Hz")
            json.dump(meta, open(meta_path, "w"), ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/testset/build_testset.py
"""Build the evaluation set: licensed voice clips mixed with synthesised instrumental music.

Music is generated procedurally (plucked strings, bass, pad, piano, drums, frame
drum and an optional voice-like lead melody), so every mixture has exact voice
and music references and nothing copyrighted is involved.

Output: data/mixtures/<id>_{voice,music}.flac (48 kHz stereo, 24-bit; mix = voice + music) and
data/mixtures/manifest.json. Also music-only items (expected output: silence).
"""
import json, os, subprocess, tempfile

import numpy as np
import soundfile as sf

SR = 48000
HERE = os.path.dirname(os.path.abspath(__file__))
SRC = os.path.join(HERE, "data", "sources")
OUT = os.path.join(HERE, "data", "mixtures")
MVR_DB = [-10.0, -3.0, 3.0]  # music-to-voice level ratios
MAX_SECONDS = 10.0


def load_48k(path):
    with tempfile.TemporaryDirectory() as d:
        dst = os.path.join(d, "o.wav")
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", path, "-ac", "1", "-ar", str(SR), "-c:a", "pcm_f32le", dst], check=True)
        x, _ = sf.read(dst, dtype="float32")
    return x


def midi(n):
    return 440.0 * 2 ** ((n - 69) / 12)


def env_adsr(n, a, d, s, r):
    e = np.ones(n) * s
    A, D, R = int(a * SR), int(d * SR), int(r * SR)
    A, D = max(A, 1), max(D, 1)
    e[:A] = np.linspace(0, 1, A)[: n] if n >= A else np.linspace(0, 1, n)
    if n > A:
        e[A:A + D] = np.linspace(1, s, D)[: max(0, min(D, n - A))]
    if R > 0 and n > R:
        e[-R:] *= np.linspace(1, 0, R)
    return e


def karplus(f, dur, rng, bright=0.5):
    n = int(dur * SR)
    p = max(int(SR / f), 2)
    buf = rng.uniform(-1, 1, p)
    out = np.empty(n)
    for i in range(n):
        out[i] = buf[i % p]
        buf[i % p] = 0.5 * (buf[i % p] + buf[(i + 1) % p]) * (0.990 + 0.009 * bright)
    return out


def saw(f, n, phase=0.0):
    t = np.arange(n) / SR
    return 2 * ((f * t + phase) % 1.0) - 1


def lowpass(x, cutoff):
    a = np.exp(-2 * np.pi * cutoff / SR)
    y = np.empty_like(x)
    acc = 0.0
    for i, v in enumerate(x):
        acc = (1 - a) * v + a * acc
        y[i] = acc
    return y


def onepole_lp_vec(x, cutoff):
    # vectorised enough for long pads: apply twice for 12 dB/oct
    from math import exp
    a = exp(-2 * np.pi * cutoff / SR)
    y = np.empty_like(x)
    acc = 0.0
    for i in range(len(x)):
        acc = (1 - a) * x[i] + a * acc
        y[i] = acc
    return y


def place(track, sig, start, pan):
    s = int(start * SR)
    e = min(s + len(sig), track.shape[0])
    if e <= s:
        return
    l, r = np.cos((pan + 1) * np.pi / 4), np.sin((pan + 1) * np.pi / 4)
    track[s:e, 0] += sig[: e - s] * l
    track[s:e, 1] += sig[: e - s] * r


def synth_music(seconds, seed, lead=True):
    rng = np.random.default_rng(seed)
    n = int(seconds * SR)
    tr = np.zeros((n, 2))
    bpm = rng.uniform(78, 128)
    beat = 60.0 / bpm
    root = int(rng.integers(45, 57))
    hijaz = rng.random() < 0.5
    scale = [0, 1, 4, 5, 7, 8, 10] if hijaz else [0, 2, 3, 5, 7, 8, 10]
    progs = [[0, 5, 3, 4], [0, 3, 4, 0], [0, 6, 5, 4], [0, 4, 5, 3]]
    prog = progs[int(rng.integers(len(progs)))]
    bars = int(np.ceil(seconds / (4 * beat))) + 1
    for b in range(bars):
        deg = prog[b % 4]
        chord = [root + scale[(deg + k) % 7] + 12 * ((deg + k) // 7) for k in (0, 2, 4)]
        t0 = b * 4 * beat
        # pad (detuned saws, slow attack)
        pn = int(4 * beat * SR)
        pad = sum(saw(midi(c + 12), pn, rng.random()) + saw(midi(c + 12) * 1.004, pn, rng.random()) for c in chord)
        pad = onepole_lp_vec(pad, 1800) * env_adsr(pn, 0.3, 0.2, 0.8, 0.4) * 0.05
        place(tr, pad, t0, rng.uniform(-0.6, 0.6))
        # bass
        for k in range(4):
            bn = int(beat * SR * 0.9)
            bs = saw(midi(chord[0] - 12), bn)
            bs = onepole_lp_vec(bs, 400) * env_adsr(bn, 0.005, 0.1, 0.6, 0.05) * 0.35
            place(tr, bs, t0 + k * beat, 0.0)
        # plucked arpeggio (guitar / oud-like)
        for k in range(8):
            note = chord[k % 3] + 12 * (k // 3 % 2) + 12
            pl = karplus(midi(note), beat * 1.5, rng, rng.random()) * 0.25
            place(tr, pl, t0 + k * beat / 2, rng.uniform(-0.8, 0.8))
        # piano-ish chord stabs on beats 1 and 3
        for k in (0, 2):
            m = int(beat * 1.8 * SR)
            t = np.arange(m) / SR
            pi_ = sum(np.sin(2 * np.pi * midi(c + 12) * h * t) / h ** 1.3 for c in chord for h in range(1, 7))
            place(tr, pi_ * np.exp(-t * 3.0) * 0.04, t0 + k * beat, rng.uniform(-0.4, 0.4))
        # drums
        for k in range(4):
            tk = t0 + k * beat
            m = int(0.35 * SR)
            t = np.arange(m) / SR
            if k % 2 == 0:  # kick
                f = 50 + 90 * np.exp(-t * 30)
                place(tr, np.sin(2 * np.pi * np.cumsum(f) / SR) * np.exp(-t * 9) * 0.6, tk, 0.0)
            else:  # snare / frame-drum slap
                sn = rng.standard_normal(m) * np.exp(-t * 18) * 0.25 + np.sin(2 * np.pi * 190 * t) * np.exp(-t * 20) * 0.2
                place(tr, sn, tk, 0.1)
            for h in range(2):  # hats
                hm = int(0.05 * SR)
                hh = np.diff(rng.standard_normal(hm + 1)) * np.exp(-np.arange(hm) / SR * 80) * 0.06
                place(tr, hh, tk + h * beat / 2, -0.3)
        # lead melody (flute/violin-like with vibrato: the voice-like hard case)
        if lead:
            for k in range(4):
                note = root + 12 + scale[int(rng.integers(7))] + 12 * int(rng.integers(0, 2))
                m = int(beat * SR)
                t = np.arange(m) / SR
                f = midi(note) * (1 + 0.006 * np.sin(2 * np.pi * 5.5 * t))
                ph = np.cumsum(f) / SR
                ld = (np.sin(2 * np.pi * ph) + 0.5 * np.sin(4 * np.pi * ph) + 0.25 * np.sin(6 * np.pi * ph))
                place(tr, ld * env_adsr(m, 0.05, 0.1, 0.8, 0.1) * 0.12, t0 + k * beat, rng.uniform(-0.3, 0.3))
    tr /= np.max(np.abs(tr)) + 1e-9
    return (tr * 0.5).astype(np.float32)


def active_rms(x, frame=1024):
    f = x[: len(x) // frame * frame].reshape(-1, frame)
    e = np.sqrt((f ** 2).mean(1) + 1e-12)
    thr = e.max() * 10 ** (-40 / 20)
    return float(np.sqrt((f[e > thr] ** 2).mean() + 1e-12))


def main():
    os.makedirs(OUT, exist_ok=True)
    meta = json.load(open(os.path.join(SRC, "SOURCES.json")))
    manifest = []
    for i, (name, m) in enumerate(sorted(meta.items())):
        v = load_48k(os.path.join(SRC, name))[: int(MAX_SECONDS * SR)]
        v = v / (np.max(np.abs(v)) + 1e-9) * 0.5
        seconds = len(v) / SR
        for j, mvr in enumerate(MVR_DB):
            seed = 1000 * i + j
            mus = synth_music(seconds, seed, lead=(j != 0))
            g = active_rms(v) * 10 ** (mvr / 20) / active_rms(mus.mean(1))
            mus = mus * g
            voice = np.stack([v, v], 1).astype(np.float32) * 0.7071
            mix = voice + mus
            peak = np.max(np.abs(mix))
            if peak > 0.98:
                voice, mus, mix = voice / peak * 0.98, mus / peak * 0.98, mix / peak * 0.98
            iid = f"{name[:-4]}_mvr{int(mvr):+d}"
            for kind, sig in (("voice", voice), ("music", mus)):  # mix = voice + music, rebuilt by the evaluator
                sf.write(os.path.join(OUT, f"{iid}_{kind}.flac"), sig, SR, subtype="PCM_24")
            manifest.append({"id": iid, "category": m["category"], "mvr_db": mvr, "lead_melody": j != 0,
                             "seconds": round(seconds, 2), "text": m.get("text", ""), "source": name})
    for k in range(5):  # music only: anything that comes out is leakage
        mus = synth_music(8.0, 90000 + k, lead=True)
        iid = f"music_only_{k}"
        sf.write(os.path.join(OUT, f"{iid}_music.flac"), mus, SR, subtype="PCM_24")
        manifest.append({"id": iid, "category": "music_only", "mvr_db": None, "lead_melody": True, "seconds": 8.0})
    json.dump(manifest, open(os.path.join(OUT, "manifest.json"), "w"), ensure_ascii=False, indent=1)
    print(f"{len(manifest)} items written to {OUT}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/eval/evaluate.py
"""Score voice-filter models on the test set by running them through the real C engine.

Usage:
  python eval/evaluate.py --candidate dfn3=models/dfn3/dfn3_stream.onnx [--candidate name=path[:mono]] [--asr]

Metrics (all on the mono sum, after removing the engine delay):
  inst_supp_db   attenuation of time-frequency bins where music dominates the voice by >10 dB (higher = fewer instruments)
  voice_ret_db   level change of bins where voice dominates music by >10 dB (0 = voice intact, negative = voice lost)
  sisdr_gain_db  SI-SDR of the output against the clean voice minus SI-SDR of the input mix
  leak_db        music-only items: output level relative to input (more negative = less leakage)
  leak_frames    music-only items: share of 100 ms frames still louder than -50 dBFS
  wer            with --asr: word error rate of the output transcript against the clean-voice transcript
"""
import argparse, json, os, subprocess, sys, tempfile, time
from collections import defaultdict

import numpy as np
import soundfile as sf

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
MIX = os.path.join(ROOT, "testset", "data", "mixtures")
CLI = os.path.join(ROOT, "build", "im-process")
SR = 48000


def stft_pow(x, n=1024, hop=512):
    w = np.hanning(n)
    frames = np.lib.stride_tricks.sliding_window_view(np.pad(x, (n, n)), n)[::hop] * w
    return np.abs(np.fft.rfft(frames, axis=1)) ** 2


def si_sdr(est, ref):
    ref = ref - ref.mean()
    est = est - est.mean()
    a = np.dot(est, ref) / (np.dot(ref, ref) + 1e-12)
    t = a * ref
    return 10 * np.log10(np.sum(t ** 2) / (np.sum((est - t) ** 2) + 1e-12) + 1e-12)


def run_engine(model, mix, mono=False):
    with tempfile.TemporaryDirectory() as d:
        i, o = os.path.join(d, "i.wav"), os.path.join(d, "o.wav")
        sf.write(i, mix, SR, subtype="FLOAT")
        cmd = [CLI, "-m", model, i, o] + (["--mono"] if mono else [])
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(r.stderr)
        y, _ = sf.read(o, dtype="float32", always_2d=True)
        info = dict(kv.split("=", 1) for kv in r.stdout.split() if "=" in kv and not kv.startswith('model'))
    return y, float(info.get("rtf", "nan"))


class Asr:
    def __init__(self, size):
        from faster_whisper import WhisperModel
        self.m = WhisperModel(size, device="cpu", compute_type="int8")

    def text(self, x, lang):
        x16 = x[::3].astype(np.float32)  # 48 kHz -> 16 kHz (inputs are band-limited enough for ASR)
        segs, _ = self.m.transcribe(x16, language=lang, beam_size=1, vad_filter=False)
        return " ".join(s.text for s in segs)


def wer(ref, hyp):
    import re
    norm = lambda s: re.sub(r"[^\w\s]", " ", s.lower()).split()
    r, h = norm(ref), norm(hyp)
    if not r:
        return float("nan")
    d = np.arange(len(h) + 1)
    for i, rw in enumerate(r, 1):
        prev, d[0] = d.copy(), i
        for j, hw in enumerate(h, 1):
            d[j] = min(prev[j] + 1, d[j - 1] + 1, prev[j - 1] + (rw != hw))
    return d[len(h)] / len(r)


def evaluate(name, model, mono, items, asr=None):
    rows, rtfs = [], []
    for it in items:
        mus = sf.read(os.path.join(MIX, f"{it['id']}_music.flac"), dtype="float32", always_2d=True)[0]
        vp = os.path.join(MIX, f"{it['id']}_voice.flac")
        voice = sf.read(vp, dtype="float32", always_2d=True)[0] if os.path.exists(vp) else None
        mix = mus + voice if voice is not None else mus
        if model == "passthrough":
            y, rtf = mix.copy(), 0.0
        else:
            y, rtf = run_engine(model, mix, mono)
        rtfs.append(rtf)
        ym, xm = y.mean(1), mix.mean(1)
        row = {"id": it["id"], "category": it["category"], "mvr_db": it["mvr_db"], "lead": it["lead_melody"]}
        if voice is None:
            row["leak_db"] = 10 * np.log10((ym ** 2).sum() / ((xm ** 2).sum() + 1e-12) + 1e-12)
            f = ym[: len(ym) // 4800 * 4800].reshape(-1, 4800)
            row["leak_frames"] = float((20 * np.log10(np.sqrt((f ** 2).mean(1)) + 1e-12) > -50).mean())
        else:
            vm, mm = voice.mean(1), mus.mean(1)
            V, M, X, Y = stft_pow(vm), stft_pow(mm), stft_pow(xm), stft_pow(ym)
            mdom = M > V * 10
            vdom = V > M * 10
            row["inst_supp_db"] = 10 * np.log10(X[mdom].sum() / (Y[mdom].sum() + 1e-12))
            row["voice_ret_db"] = 10 * np.log10(Y[vdom].sum() / (X[vdom].sum() + 1e-12))
            row["sisdr_gain_db"] = si_sdr(ym, vm) - si_sdr(xm, vm)
            if asr is not None:
                lang = "en" if it["category"] == "speech_en" else "ar"
                if it["category"] != "singing":
                    ref = asr.text(vm, lang)
                    row["wer"] = wer(ref, asr.text(ym, lang))
                    row["wer_mix"] = wer(ref, asr.text(xm, lang))
        rows.append(row)
        print(f"  {name} {it['id']}: " + ", ".join(f"{k}={v:.2f}" for k, v in row.items() if isinstance(v, float)), flush=True)
    return rows, float(np.nanmean(rtfs)) if rtfs else 0.0


def summarise(name, rows, rtf):
    groups = defaultdict(list)
    for r in rows:
        groups[r["category"]].append(r)
        if r["mvr_db"] is not None:
            groups[f"all voice, music {r['mvr_db']:+.0f} dB"].append(r)
    keys = ["inst_supp_db", "voice_ret_db", "sisdr_gain_db", "wer", "wer_mix", "leak_db", "leak_frames"]
    out = {}
    for g, rs in sorted(groups.items()):
        out[g] = {k: round(float(np.nanmean([r[k] for r in rs if k in r])), 2) for k in keys if any(k in r for r in rs)}
        out[g]["n"] = len(rs)
    return {"candidate": name, "rtf_stereo_1thread": round(rtf, 3), "groups": out}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--candidate", action="append", default=[], help="name=model.onnx[:mono] or name=passthrough")
    ap.add_argument("--asr", nargs="?", const="base", help="also score word error rate with faster-whisper (model size)")
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--out", default=os.path.join(ROOT, "eval", "results"))
    args = ap.parse_args()
    items = json.load(open(os.path.join(MIX, "manifest.json")))
    if args.limit:
        items = items[: args.limit]
    asr = Asr(args.asr) if args.asr else None
    os.makedirs(args.out, exist_ok=True)
    cands = args.candidate or ["passthrough=passthrough", f"dfn3={os.path.join(ROOT, 'models/dfn3/dfn3_stream.onnx')}"]
    summary = []
    for c in cands:
        name, spec = c.split("=", 1)
        mono = spec.endswith(":mono")
        model = spec[:-5] if mono else spec
        t0 = time.time()
        rows, rtf = evaluate(name, model, mono, items, asr)
        s = summarise(name, rows, rtf)
        s["seconds"] = round(time.time() - t0, 1)
        json.dump({"summary": s, "rows": rows}, open(os.path.join(args.out, f"{name}.json"), "w"), indent=1)
        summary.append(s)
    lines = ["| candidate | group | n | instrument suppression dB | voice retention dB | SI-SDR gain dB | WER out | WER mix | music-only leak dB | leak frames |", "|---|---|---|---|---|---|---|---|---|---|"]
    for s in summary:
        for g, v in s["groups"].items():
            f = lambda k: f"{v[k]:.2f}" if k in v else ""
            lines.append(f"| {s['candidate']} | {g} | {v['n']} | {f('inst_supp_db')} | {f('voice_ret_db')} | {f('sisdr_gain_db')} | {f('wer')} | {f('wer_mix')} | {f('leak_db')} | {f('leak_frames')} |")
    lines.append("")
    for s in summary:
        lines.append(f"- {s['candidate']}: real-time factor {s['rtf_stereo_1thread']} (stereo, one thread), eval took {s['seconds']} s")
    open(os.path.join(args.out, "summary.md"), "w").write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/kaggle/prepare_data.py
"""Prepare training audio on Kaggle: decode once to 48 kHz mono 16-bit WAV clips (fast random reads).

Sources (licences in models/voice/README.md):
  voice/speech_en   LibriSpeech (attached Kaggle dataset; test-clean/test-other excluded)      CC BY 4.0
  voice/speech_ar   Arabic Speech Corpus (Hugging Face tunis-ai/arabic_speech_corpus)           CC BY 4.0
  voice/singing     VocalSet (Hugging Face Bill13579/vocalset-mirror)                           CC BY 4.0
  voice/recitation  Diverse Quran Reciters (attached Kaggle dataset)          rights belong to the reciters
  music/slakh       Slakh2100 mixes (attached Kaggle dataset) or the 1 GB stems+mixes subset tvergho/slakh   CC BY 4.0
Clips used by our evaluation set (testset/fetch_sources.py) are skipped.

Usage: python prepare_data.py --out /kaggle/tmp/data [--hours speech_en=3,speech_ar=3,singing=4,recitation=4,slakh=10]
"""
import argparse, glob, io, os, random, subprocess, sys, tempfile
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import soundfile as sf

SR = 48000
CLIP_S = 20
EXCLUDE = {  # rows used by the evaluation set
    "speech_ar": {0, 50, 100, 150},
    "singing": {0, 200, 900, 1500, 2300, 3100},
}


def decode(src, data=None):
    """ffmpeg -> mono float32 at 48 kHz. src is a path, or a name hint when data (bytes) is given."""
    if data is not None:
        with tempfile.NamedTemporaryFile(suffix=os.path.splitext(src)[1] or ".bin") as t:
            t.write(data)
            t.flush()
            return decode(t.name)
    out = subprocess.run(["ffmpeg", "-v", "error", "-i", src, "-ac", "1", "-ar", str(SR), "-f", "f32le", "-"],
                         capture_output=True)
    return np.frombuffer(out.stdout, np.float32).copy() if out.returncode == 0 else np.zeros(0, np.float32)


def trim(x, thr_db=-50):
    if not len(x):
        return x
    e = np.abs(x)
    idx = np.where(e > np.max(e) * 10 ** (thr_db / 20))[0]
    return x[idx[0]:idx[-1] + 1] if len(idx) else x[:0]


def write_clips(x, out_dir, stem):
    """Split into <= CLIP_S second clips, normalise peak, save as 16-bit WAV. Returns seconds written."""
    x = trim(x)
    if len(x) < SR:  # skip < 1 s
        return 0.0
    os.makedirs(out_dir, exist_ok=True)
    total = 0.0
    for k, s in enumerate(range(0, len(x), CLIP_S * SR)):
        c = x[s:s + CLIP_S * SR]
        if len(c) < SR:
            continue
        c = c / (np.max(np.abs(c)) + 1e-9) * 0.9
        sf.write(os.path.join(out_dir, f"{stem}_{k}.wav"), c, SR, subtype="PCM_16")
        total += len(c) / SR
    return total


def job_file(args):
    path, out_dir, stem = args
    return write_clips(decode(path), out_dir, stem)


INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")


def find(pattern, hint):
    """Search only inside attached datasets whose folder name contains `hint`
    (works for both /kaggle/input/<slug> and /kaggle/input/datasets/<owner>/<slug> layouts)."""
    roots = [d for d in glob.glob(f"{INPUT}/*") + glob.glob(f"{INPUT}/*/*/*") if hint in os.path.basename(d).lower()]
    out = set()
    for r in roots:
        out.update(glob.glob(os.path.join(r, "**", pattern), recursive=True))
    print(f"found {len(out)} {pattern} files for '{hint}' in {roots}", flush=True)
    return sorted(out)


def from_files(files, out_dir, budget_s, workers, rng, prefix):
    rng.shuffle(files)
    done, i = 0.0, 0
    with ProcessPoolExecutor(workers) as ex:
        while done < budget_s and i < len(files):
            batch = files[i:i + workers * 4]
            i += len(batch)
            jobs = [(f, out_dir, f"{prefix}{i + k}") for k, f in enumerate(batch)]
            done += sum(ex.map(job_file, jobs))
    return done


def from_hf_parquet(repo, out_dir, budget_s, exclude, rng, split_hint="train", columns=("audio",)):
    """Stream rows from a Hugging Face parquet dataset (audio columns with bytes)."""
    import pyarrow.parquet as pq
    from huggingface_hub import HfApi, hf_hub_download
    files = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset")
                   if f.endswith(".parquet") and split_hint in f)
    done, row = 0.0, 0
    for f in files:
        local = hf_hub_download(repo, f, repo_type="dataset")
        table = pq.ParquetFile(local)
        for batch in table.iter_batches(batch_size=32, columns=list(columns)):
            cols = [batch.column(i).to_pylist() for i in range(len(columns))]
            for vals in zip(*cols):
                r, row = row, row + 1
                if r in exclude:
                    continue
                for ci, a in enumerate(vals):
                    if not a or not a.get("bytes"):
                        continue
                    done += write_clips(decode(a.get("path") or "x.wav", a["bytes"]), out_dir, f"r{r}c{ci}")
                if done >= budget_s:
                    os.remove(local)
                    return done
        os.remove(local)  # keep disk use low
    return done


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="/kaggle/tmp/data")
    ap.add_argument("--hours", default="speech_en=3,speech_ar=3,singing=4,recitation=4,slakh=10")
    ap.add_argument("--no-quran", action="store_true", help="skip recitation data (rights not cleared)")
    ap.add_argument("--workers", type=int, default=os.cpu_count() or 4)
    args = ap.parse_args()
    hours = {k: float(v) for k, v in (kv.split("=") for kv in args.hours.split(","))}
    rng = random.Random(0)
    report = {}

    libri = [f for f in find("*.flac", "librispeech") if "/test-" not in f]
    report["speech_en"] = from_files(libri, f"{args.out}/voice/speech_en", hours["speech_en"] * 3600, args.workers, rng, "ls")

    report["speech_ar"] = from_hf_parquet("tunis-ai/arabic_speech_corpus", f"{args.out}/voice/speech_ar",
                                          hours["speech_ar"] * 3600, EXCLUDE["speech_ar"], rng)
    report["singing"] = from_hf_parquet("Bill13579/vocalset-mirror", f"{args.out}/voice/singing",
                                        hours["singing"] * 3600, EXCLUDE["singing"], rng)
    if not args.no_quran:
        quran = find("*.mp3", "quran")
        report["recitation"] = from_files(quran, f"{args.out}/voice/recitation", hours["recitation"] * 3600,
                                          args.workers, rng, "q")

    # music: full Slakh mixes plus random partial sub-mixes (fewer instruments = harder, more varied)
    mixes = [f for f in find("mix.flac", "slakh") if "/omitted/" not in f and "/test/" not in f]
    if mixes:  # full Slakh2100 attached as a Kaggle dataset
        report["slakh"] = from_files(mixes, f"{args.out}/music/slakh", hours["slakh"] * 3600, args.workers, rng, "s")
    else:  # 1 GB Slakh subset on Hugging Face: single instrument stems plus their mixes
        report["slakh"] = from_hf_parquet("tvergho/slakh", f"{args.out}/music/slakh", hours["slakh"] * 3600, set(),
                                          rng, split_hint="test", columns=("audio", "stem_mix"))

    for k, v in report.items():
        print(f"{k:12s} {v / 3600:6.2f} h")
    missing = [k for k, v in report.items() if v < 600]
    if missing:
        sys.exit(f"too little data for: {missing} (check the attached datasets)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/kaggle/eval_onnx.py
"""Score streaming ONNX voice models on the evaluation set with ONNX Runtime (Python version of
eval/evaluate.py, for machines without the C engine, e.g. Kaggle). Same metrics, same test set.

Usage: python kaggle/eval_onnx.py name=model.onnx [name2=model2.onnx ...]
"""
import json, os, sys, time
from collections import defaultdict

import numpy as np
import onnxruntime as ort
import soundfile as sf

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.path.join(ROOT, "eval"))
from evaluate import MIX, si_sdr, stft_pow  # noqa: E402


def load(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    s = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    m = s.get_modelmeta().custom_metadata_map
    init_p = path[:-5] + ".state_init.bin"
    init = np.fromfile(init_p, "<f4") if os.path.exists(init_p) else np.zeros(int(m["im.state_size"]), np.float32)
    return s, int(m["im.hop"]), int(m["im.latency_samples"]), init


def run(model, x):
    sess, hop, lat, init = model
    T, C = x.shape
    pad = np.concatenate([x, np.zeros((lat + hop, C), np.float32)])
    out = np.zeros_like(pad)
    for c in range(C):
        st = init.copy()
        for k in range(0, len(pad) - hop + 1, hop):
            y, st = sess.run(None, {"audio_in": np.ascontiguousarray(pad[k:k + hop, c]), "state_in": st})
            out[k:k + hop, c] = y
    return out[lat:lat + T]


def score(model, items):
    rows = []
    for it in items:
        mus = sf.read(os.path.join(MIX, f"{it['id']}_music.flac"), dtype="float32", always_2d=True)[0]
        vp = os.path.join(MIX, f"{it['id']}_voice.flac")
        voice = sf.read(vp, dtype="float32", always_2d=True)[0] if os.path.exists(vp) else None
        mix = mus + voice if voice is not None else mus
        y = run(model, mix).mean(1)
        xm = mix.mean(1)
        r = {"id": it["id"], "category": it["category"], "mvr_db": it["mvr_db"]}
        if voice is None:
            r["leak_db"] = float(10 * np.log10((y ** 2).sum() / ((xm ** 2).sum() + 1e-12) + 1e-12))
        else:
            vm, mm = voice.mean(1), mus.mean(1)
            V, M, X, Y = stft_pow(vm), stft_pow(mm), stft_pow(xm), stft_pow(y)
            r["inst_supp_db"] = float(10 * np.log10(X[M > V * 10].sum() / (Y[M > V * 10].sum() + 1e-12)))
            r["voice_ret_db"] = float(10 * np.log10(Y[V > M * 10].sum() / (X[V > M * 10].sum() + 1e-12)))
            r["sisdr_gain_db"] = float(si_sdr(y, vm) - si_sdr(xm, vm))
        rows.append(r)
    return rows


def main():
    items = json.load(open(os.path.join(MIX, "manifest.json")))
    lines = ["| model | group | n | instrument suppression dB | voice retention dB | SI-SDR gain dB | music-only leak dB |",
             "|---|---|---|---|---|---|---|"]
    results = {}
    for spec in sys.argv[1:]:
        name, path = spec.split("=", 1)
        t0 = time.time()
        rows = score(load(path), items)
        results[name] = rows
        groups = defaultdict(list)
        for r in rows:
            groups[r["category"]].append(r)
        for g, rs in sorted(groups.items()):
            m = lambda k: f"{np.mean([r[k] for r in rs if k in r]):.2f}" if any(k in r for r in rs) else ""
            lines.append(f"| {name} | {g} | {len(rs)} | {m('inst_supp_db')} | {m('voice_ret_db')} | {m('sisdr_gain_db')} | {m('leak_db')} |")
        print(f"{name}: scored in {time.time() - t0:.0f} s", flush=True)
    text = "\n".join(lines)
    print(text)
    out = os.environ.get("EVAL_OUT", ".")
    open(os.path.join(out, "eval_summary.md"), "w").write(text + "\n")
    json.dump(results, open(os.path.join(out, "eval_rows.json"), "w"), indent=1)


if __name__ == "__main__":
    main()

## 1. Isolated Python 3.10 environment
Same versions as the verified local setup (torch 2.1.2, ONNX Runtime 1.19.2, numpy 1.26, deepfilternet 0.5.6),
so the official DeepFilterNet wheels work regardless of Kaggle's default Python.

In [ ]:
PY = f"{TMP}/venv/bin/python"
if not os.path.exists(PY):
    run("pip install -q uv")
    run(f"uv venv -q -p 3.10 {TMP}/venv")
    run(f"uv pip install -q -p {PY} torch==2.1.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121")
    run(f"uv pip install -q -p {PY} deepfilternet==0.5.6 numpy==1.26.4 soundfile onnx==1.16.2 onnxruntime==1.19.2 "
        "'protobuf<5' huggingface_hub pyarrow")
run(f"{PY} -c 'import torch; print(\"torch\", torch.__version__, \"cuda\", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"\")'")
run("command -v ffmpeg || (apt-get update -qq && apt-get install -y -qq ffmpeg)")
run("ffmpeg -version | head -1")
run(f"{PY} test_frontend.py", cwd=f"{CODE}/models/voice")   # training front end == libDF

## 2. Prepare training audio

In [ ]:
args = f"--out {TMP}/data --hours {DATA_HOURS}" + ("" if USE_QURAN else " --no-quran")
run(f"{PY} kaggle/prepare_data.py {args}")
run(f"du -sh {TMP}/data/*/*")

## 3. Train (smoke test first, then the full run with a time budget)

In [ ]:
voice = [f"--voice {TMP}/data/voice/{k}:{w}" for k, w in VOICE_WEIGHTS.items() if os.path.isdir(f"{TMP}/data/voice/{k}")]
common = " ".join(voice) + f" --music {TMP}/data/music/slakh --synth-music 0.15 --workers 4"
run(f"{PY} models/voice/train.py {common} --steps 20 --batch 4 --log-every 5 --out {TMP}/smoke")
t0 = time.time()
run(f"{PY} models/voice/train.py {common} --steps {MAX_STEPS} --batch {BATCH} --hours {TRAIN_HOURS} "
    f"--log-every 200 --save-every 1000 --out {OUT}")
print(f"training took {(time.time() - t0) / 3600:.2f} h")

## 4. Export streaming ONNX (baseline and fine-tuned)

In [ ]:
run(f"{PY} models/dfn3/export_stream.py --out {OUT}/dfn3_baseline.onnx")
run(f"{PY} models/dfn3/export_stream.py --checkpoint {OUT}/last.pt --out {OUT}/voice_v1.onnx "
    "--name 'Voice v1 (speech, recitation, singing)' --keeps speech,recitation,singing")

## 5. Evaluate on the held-out test set

In [ ]:
run(f"{PY} testset/fetch_sources.py")
run(f"{PY} testset/build_testset.py")
run(f"{PY} kaggle/eval_onnx.py baseline={OUT}/dfn3_baseline.onnx voice_v1={OUT}/voice_v1.onnx", env={"EVAL_OUT": OUT})

In [ ]:
run(f"rm -rf {CODE}/testset/data; ls -la {OUT}")
print(open(f"{OUT}/eval_summary.md").read())